In [1]:
!pip install tensorflow

In [2]:
!pip install scikeras

In [3]:
!pip install --upgrade ml_dtypes

In [4]:
!pip install "scikit-learn<1.6" --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 117.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [5]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras

In [6]:
import time
from scikeras.wrappers import KerasRegressor
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import metrics

In [7]:
print('scikeras:', scikeras.__version__)
print('scikit-learn:', sklearn.__version__)
print('tensorflow:', tf.__version__)

scikeras: 0.13.0
scikit-learn: 1.5.2
tensorflow: 2.20.0


In [8]:
inicio = time.time()

In [9]:
inicio

1786040916.616832

In [10]:
base = pd.read_csv('/content/sample_data/autos.csv', encoding='ISO-8859-1')
# le o csv

In [11]:
base = base.drop('dateCrawled', axis=1)
base = base.drop('dateCreated', axis=1)
base = base.drop('nrOfPictures', axis=1)
base = base.drop('postalCode', axis=1)
base = base.drop('lastSeen', axis=1)
base = base.drop('name', axis=1)
base = base.drop('seller', axis=1)
base = base.drop('offerType', axis=1)
# remove todas as colunas que nao servem para o modelo prever o preço

In [12]:
base = base[base.price > 10]
# tira valores muito baixos
base = base.loc[base.price < 350000]
# tira os outliers

In [13]:
valores = {'vehicleType': 'limousine',
           'gearbox': 'manuell',
           'model': 'golf',
           'fuelType': 'benzin',
           'notRepairedDamage': 'nein'}
base = base.fillna(value=valores)
# suistitui cada valor nullo com os valores que mais aparecem em cada feature

In [14]:
x = base.iloc[:, 1:12].values
# pega todas as linhas e colunas menos o preço
y = base.iloc[:, 0].values
# pega somente a coluna preço

In [15]:
x, y

(array([['test', 'limousine', 1993.0, ..., 'benzin', 'volkswagen', 'nein'],
        ['test', 'coupe', 2011.0, ..., 'diesel', 'audi', 'ja'],
        ['test', 'suv', 2004.0, ..., 'diesel', 'jeep', 'nein'],
        ...,
        ['control', 'limousine', 2018.0, ..., 'benzin', 'volkswagen',
         'nein'],
        ['control', 'limousine', 2018.0, ..., 'benzin', 'opel', 'nein'],
        ['test', 'kleinwagen', 2002.0, ..., 'benzin', 'opel', 'nein']],
       dtype=object),
 array([  480., 18300.,  9800., ...,  1190.,  1350.,   500.]))

In [16]:
onehotencoder = ColumnTransformer(transformers=[("OneHot", OneHotEncoder(), [0, 1, 3, 5, 8, 9, 10])], remainder='passthrough')
# o onehotencoder vai transformar todos os valores de colunas categoricas para uma codificação propria, diferente do labelencoder que classific com ordem de maior valor
x = onehotencoder.fit_transform(x).toarray()

In [17]:
x.shape

(324740, 316)

In [18]:
def criar_rede():
    k.clear_session()
    regressor = Sequential([
        tf.keras.layers.InputLayer(shape=(316,)),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=158, activation='relu'),
        tf.keras.layers.Dense(units=1, activation='linear')])
    regressor.compile(loss='mean_absolute_error', optimizer='adam', metrics=['mean_absolute_error'])
    return regressor
# cria a rede neural com 316 entradas para uma saida sendo ela numerica
# o mean_absolute_error é o mais usado em problemas de regressao pois ele mede a diferença do valor de predição do modelo para os valores reais

In [19]:
regressor = KerasRegressor(model = criar_rede, epochs = 100, batch_size = 300)
# cria a rede com o modelo e parametros

In [20]:
resultados = cross_val_score(estimator = regressor, X = x, y = y,cv = 5, scoring = 'neg_mean_absolute_error')
#o cross validation vai testar diferentes conjuntos de dados para treinar evitando assim a divisao perder algum valor determinante

Epoch 1/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 3943.9097 - mean_absolute_error: 3943.9097
Epoch 2/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 3436.1006 - mean_absolute_error: 3436.1006
Epoch 3/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3286.3145 - mean_absolute_error: 3286.3145
Epoch 4/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 3091.8040 - mean_absolute_error: 3091.8040
Epoch 5/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2933.2971 - mean_absolute_error: 2933.2971
Epoch 6/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 2870.8220 - mean_absolute_error: 2870.8220
Epoch 7/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 2818.0693 - mean_absolute_error: 2818.0693
Epoch 8/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2783.3918 - mean_absolute_error: 2783.3918
Epoch 9/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2745.2205 - mean_absolute_error: 2745.2205
Epoch 10/100
866/866 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 2

In [21]:
fim = time.time()

In [22]:
fim

1786042260.0722768

In [23]:
(fim - inicio) / 60 / 60

0.37318206800354853

In [24]:
abs(resultados)
# pega a 'distancia dos 5 testes realizados'

array([2279.97847989, 2254.20822049, 2216.77481111, 2227.67049104,
       2309.09703963])

In [25]:
abs(resultados.mean())
# faz a media dos valores abslutos

np.float64(2257.545808431634)

In [26]:
resultados.std()# desvio padrao

np.float64(33.84287958083335)